# Fine-tune T5-small on GYAFC (informal → formal)
Train a model to rewrite informal English sentences into formal language while preserving meaning.

**In Colab:** choose **Runtime → Change runtime type → GPU**, then run the cells in order. Section 3 asks for the GYAFC corpus, which is **not** on the Hugging Face Hub and must be supplied from your own licensed copy.

Set `SMOKE_TEST = True` in the configuration cell for a quick end-to-end check before full training. A smoke-test model is only a debugging artifact, not a useful trained model.

GYAFC is distributed under a Yahoo/Grammarly research license. Do not commit the corpus to this repository, and do not redistribute it.

Sources: [GYAFC (Rao & Tetreault, 2018)](https://aclanthology.org/N18-1012/), [T5-small](https://huggingface.co/google-t5/t5-small), and [Hugging Face sequence-to-sequence training](https://huggingface.co/docs/transformers/tasks/translation).

## 1. Install dependencies
Use a fresh Colab runtime with Python 3.10 or newer. Colab provides PyTorch; install the remaining packages below. If Colab asks to restart the session, restart it and continue from the next cell. These versions use `eval_strategy` and `processing_class` in the Trainer API.

`sacrebleu` is used only by the optional BLEU cell in section 9.

In [1]:
%pip install -q "transformers==4.57.3" "datasets==4.4.1" "accelerate==1.12.0" "sentencepiece==0.2.1" "sacrebleu==2.4.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Configure the run
These are the main settings to edit. The effective training batch size on one GPU is `8 × 2 = 16`. Full precision is used for both training and evaluation. All model parameters will be trained.

GYAFC ships its own train/tune/test splits, so this notebook uses them directly rather than resampling. `tune` becomes the validation split.

The `tune` and `test` folders are **cross-named**: the source sentences are in `informal`, and their four human formal references are `formal.ref0`–`formal.ref3`. (`informal.ref0`–`informal.ref3` are references for the reverse formal→informal task and are ignored here.)

`MAX_LENGTH = 64` covers every validation and test sentence — the longest is 58 tokens — and all but 20 of the 104,562 training sources. The sources it clips are degenerate strings, such as a six-word sentence padded with 160 question marks. Because padding is dynamic, this is a guard against those rows, not a speed setting.

Smoke mode uses at most 128 training rows, 32 validation rows, 32 test rows, and 5 optimizer steps. Its output folders are separate from the full model.

In [2]:
import os
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

MODEL_NAME = "google-t5/t5-small"
DATASET_NAME = "GYAFC"  # Local line-aligned corpus; not available on the Hugging Face Hub.
DOMAINS = ("Entertainment_Music", "Family_Relationships")
SPLIT_DIRS = {"train": "train", "validation": "tune", "test": "test"}
SOURCE_COLUMN = "informal"
TARGET_COLUMN = "formal"
REFERENCE_COUNT = 4          # formal.ref0 .. formal.ref3 in tune/ and test/
LABEL_REFERENCE_INDEX = 0    # Which human reference supplies the eval/test loss label.
DROP_TRAIN_OVERLAP = True    # Remove the few tune/test sources that also appear in train.
DROP_IDENTITY_PAIRS = False  # Keep train pairs whose formal side equals the informal side.
PREFIX = "formalize: "
GYAFC_DIRNAME = "GYAFC_Corpus"
EXTRACT_ROOT = Path("/content") if Path("/content").is_dir() else Path(".")
SEED = 42
MAX_LENGTH = 64
EPOCHS = 3
LEARNING_RATE = 3e-4
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
SMOKE_TEST = False
SMOKE_STEPS = 5

RUN_DIR = "./t5-small-gyafc-formal-smoke-checkpoints" if SMOKE_TEST else "./t5-small-gyafc-formal-checkpoints"
EXPORT_DIR = "./t5-small-gyafc-formal-smoke" if SMOKE_TEST else "./t5-small-gyafc-formal"
os.environ["WANDB_DISABLED"] = "true"
set_seed(SEED)
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Mode:", "smoke test" if SMOKE_TEST else "full training")
if not torch.cuda.is_available():
    print("For Colab training, switch to a GPU runtime. CPU training will be much slower.")

CUDA GPU: NVIDIA A100-SXM4-80GB
Mode: full training


## 3. Provide the GYAFC corpus
GYAFC is license-restricted and is not on the Hugging Face Hub, so `load_dataset("name")` cannot fetch it. **Upload `GYAFC_Corpus.zip` to the Colab session at `/content` before running the cell below.** The cell searches the working directory and `/content` first, extracts the archive, and never prompts for an upload; if it finds nothing it lists every path it checked.

`/content` is wiped when the runtime recycles, so re-upload the archive after a restart. It is about 6.5 MB.

Extraction rejects absolute paths, members that escape the target folder, and symlinks, and it skips the `__MACOSX/` resource forks that the official archive contains.

In [4]:
SEARCH_DIRS = (
    Path("."),            # The Colab working directory is /content.
    Path("/content"),
    Path("/content/data"),
    Path("data"),
    Path("..") / "data",
)

def looks_like_corpus(path):
    """True only if the expected data files are present, not just the folder."""
    return all((path / domain / "train" / "informal").is_file() for domain in DOMAINS)


def safe_members(archive, target):
    """Yield only GYAFC_Corpus/* members, rejecting escaping paths and symlinks."""
    for member in archive.infolist():
        name = member.filename
        if name.startswith("__MACOSX/") or Path(name).name.startswith("._"):
            continue  # macOS resource forks in the official archive.
        if name != f"{GYAFC_DIRNAME}/" and not name.startswith(f"{GYAFC_DIRNAME}/"):
            raise ValueError(f"Unexpected ZIP member outside {GYAFC_DIRNAME}/: {name}")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f"Symlink in archive: {name}")
        destination = (EXTRACT_ROOT / name).resolve()
        if destination != target and not destination.is_relative_to(target):
            raise ValueError(f"Unexpected ZIP member path: {name}")
        yield member


def extract_corpus(archive_path):
    target = (EXTRACT_ROOT / GYAFC_DIRNAME).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(EXTRACT_ROOT, members=list(safe_members(archive, target)))
    return target


def resolve_corpus():
    for directory in SEARCH_DIRS:
        candidate = directory / GYAFC_DIRNAME
        if looks_like_corpus(candidate):
            return candidate.resolve()
    for directory in SEARCH_DIRS:
        archive = directory / f"{GYAFC_DIRNAME}.zip"
        if archive.is_file():
            print("Extracting", archive.resolve())
            extracted = extract_corpus(archive)
            if looks_like_corpus(extracted):
                return extracted
    searched = "\n  ".join(str(directory.resolve()) for directory in SEARCH_DIRS)
    raise FileNotFoundError(
        f"Could not find {GYAFC_DIRNAME}/ or {GYAFC_DIRNAME}.zip in:\n  {searched}\n"
        f"Upload {GYAFC_DIRNAME}.zip to /content, then re-run this cell."
    )


GYAFC_ROOT = resolve_corpus()
print("Corpus root:", GYAFC_ROOT)
for domain in DOMAINS:
    for directory in SPLIT_DIRS.values():
        assert (GYAFC_ROOT / domain / directory).is_dir(), f"Missing {domain}/{directory}"
print("Found both domains with train/tune/test.")

Corpus root: /content/GYAFC_Corpus
Found both domains with train/tune/test.


## 4. Load the parallel line files
Each GYAFC split is a set of plain UTF-8, line-aligned text files with no extensions, so the cell below reads them directly and builds a `DatasetDict` with `Dataset.from_dict`.

Both domains are concatenated, with a `domain` column so counts can be reported per domain. Line counts are asserted equal across every file in a split **before** any row is built — a line-aligned corpus fails silently otherwise, pairing each sentence with the wrong rewrite. Rows are dropped only when the source or any reference is blank after stripping, and always as a whole row so alignment is preserved.

Train has one crowdsourced formal rewrite per sentence. Validation and test keep all four human references in a `references` column, and reference `LABEL_REFERENCE_INDEX` supplies the `formal` column that `eval_loss` and `test_loss` score against. The four references are equally valid rewrites in different styles; index 0 is a fixed arbitrary choice, not the best one.

In [5]:
def read_lines(path):
    text = Path(path).read_text(encoding="utf-8")
    lines = text.splitlines()
    # Every GYAFC file ends in a newline and has no interior blank lines. splitlines()
    # also breaks on \x0b \x0c \x85 \u2028 \u2029, so cross-check against newlines.
    assert len(lines) == text.count("\n"), f"{path}: unexpected line breaks or no final newline"
    return lines


def read_domain_split(domain, directory):
    base = GYAFC_ROOT / domain / directory
    sources = read_lines(base / SOURCE_COLUMN)
    if directory == "train":
        reference_files = [TARGET_COLUMN]
    else:
        # tune/ and test/ are cross-named: informal is the source, formal.ref* its references.
        reference_files = [f"{TARGET_COLUMN}.ref{index}" for index in range(REFERENCE_COUNT)]
    references = [read_lines(base / name) for name in reference_files]
    for name, column in zip(reference_files, references):
        assert len(column) == len(sources), (
            f"{domain}/{directory}: {name} has {len(column):,} lines, "
            f"but {SOURCE_COLUMN} has {len(sources):,}"
        )
    label_index = 0 if directory == "train" else LABEL_REFERENCE_INDEX
    rows, dropped = [], 0
    for position, source in enumerate(sources):
        source = source.strip()
        candidates = [column[position].strip() for column in references]
        if not source or not all(candidates):
            dropped += 1
            continue
        rows.append({
            SOURCE_COLUMN: source,
            TARGET_COLUMN: candidates[label_index],
            "references": candidates,
            "domain": domain,
        })
    print(f"{domain}/{directory}: {len(rows):,} rows from {len(sources):,} lines ({dropped} dropped as blank).")
    return rows


split_rows = {
    name: [row for domain in DOMAINS for row in read_domain_split(domain, directory)]
    for name, directory in SPLIT_DIRS.items()
}


def input_key(text):
    return " ".join(text.split()).casefold()


train_keys = {input_key(row[SOURCE_COLUMN]) for row in split_rows["train"]}
for name in ("validation", "test"):
    overlapping = sum(1 for row in split_rows[name] if input_key(row[SOURCE_COLUMN]) in train_keys)
    print(f"{name}: {overlapping} of {len(split_rows[name]):,} sources also appear in train.")
    if DROP_TRAIN_OVERLAP:
        split_rows[name] = [
            row for row in split_rows[name] if input_key(row[SOURCE_COLUMN]) not in train_keys
        ]

identity = sum(
    1 for row in split_rows["train"]
    if input_key(row[SOURCE_COLUMN]) == input_key(row[TARGET_COLUMN])
)
print(f"train: {identity:,} pairs whose formal side matches the informal side after normalization.")
if DROP_IDENTITY_PAIRS:
    split_rows["train"] = [
        row for row in split_rows["train"]
        if input_key(row[SOURCE_COLUMN]) != input_key(row[TARGET_COLUMN])
    ]


def to_dataset(rows):
    assert rows, "A split ended up empty."
    return Dataset.from_dict({column: [row[column] for row in rows] for column in rows[0]})


splits = DatasetDict({name: to_dataset(rows) for name, rows in split_rows.items()})
if SMOKE_TEST:
    limits = {"train": 128, "validation": 32, "test": 32}
    splits = DatasetDict({
        name: data.shuffle(seed=SEED).select(range(min(len(data), limits[name])))
        for name, data in splits.items()
    })
print(splits)
for name, data in splits.items():
    print(name, "domains:", dict(Counter(data["domain"])))
print("\ntrain example source:", splits["train"][0][SOURCE_COLUMN])
print("train example target:", splits["train"][0][TARGET_COLUMN])
test_example = splits["test"][0]
print("\ntest example source: ", test_example[SOURCE_COLUMN])
for index, reference in enumerate(test_example["references"]):
    print(f"test example ref{index}:  ", reference)
# Sanity check the cross-naming: these four must be formal rewrites of the source above.
assert test_example[TARGET_COLUMN] == test_example["references"][LABEL_REFERENCE_INDEX]

Entertainment_Music/train: 52,595 rows from 52,595 lines (0 dropped as blank).
Family_Relationships/train: 51,967 rows from 51,967 lines (0 dropped as blank).
Entertainment_Music/tune: 2,877 rows from 2,877 lines (0 dropped as blank).
Family_Relationships/tune: 2,788 rows from 2,788 lines (0 dropped as blank).
Entertainment_Music/test: 1,416 rows from 1,416 lines (0 dropped as blank).
Family_Relationships/test: 1,332 rows from 1,332 lines (0 dropped as blank).
validation: 10 of 5,665 sources also appear in train.
test: 27 of 2,748 sources also appear in train.
train: 1,200 pairs whose formal side matches the informal side after normalization.
DatasetDict({
    train: Dataset({
        features: ['informal', 'formal', 'references', 'domain'],
        num_rows: 104562
    })
    validation: Dataset({
        features: ['informal', 'formal', 'references', 'domain'],
        num_rows: 5655
    })
    test: Dataset({
        features: ['informal', 'formal', 'references', 'domain'],
        

## 5. Tokenize inputs and targets
T5 learns a text-to-text task: `formalize: <informal sentence>` → `<formal rewrite>`. Use the same prefix at inference time — it is a learned task token, not one of T5's native pretraining prefixes, so the exported model is useless without the exact string.

Tokenize inputs and targets separately and truncate each to 64 tokens. Do not pad the whole dataset here. The data collator pads each batch to its longest sequence and uses `-100` for target padding, which the loss ignores.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
assert all(parameter.requires_grad for parameter in model.parameters())

def tokenize_batch(batch):
    inputs = [PREFIX + text for text in batch[SOURCE_COLUMN]]
    encoded = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
    targets = tokenizer(text_target=batch[TARGET_COLUMN], max_length=MAX_LENGTH, truncation=True)
    encoded["labels"] = targets["input_ids"]
    return encoded

tokenized = splits.map(tokenize_batch, batched=True, remove_columns=splits["train"].column_names)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Tokenized fields:", tokenized["train"].column_names)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/104562 [00:00<?, ? examples/s]

Map:   0%|          | 0/5655 [00:00<?, ? examples/s]

Map:   0%|          | 0/2721 [00:00<?, ? examples/s]

Trainable parameters: 60,506,624
Tokenized fields: ['input_ids', 'attention_mask', 'labels']


## 6. Train and keep the best checkpoint
For a full run, evaluate and save after every epoch. The lowest validation loss determines the best checkpoint, which is loaded automatically at the end. Keep at most two checkpoints to limit disk usage.

104,562 training rows at an effective batch of 16 give about 6,535 optimizer steps per epoch, so three epochs is roughly 19,605 steps and about 45 minutes on an A100 — against 4,935 steps and about 10 minutes for the ParaDetox run on a much smaller dataset.

In smoke mode, evaluate and save after the last of five steps instead. Checkpoints are separate from the final exported model. Re-running training starts a new run; use a new `RUN_DIR` if you want to preserve old checkpoints.

In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir=RUN_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy="steps" if SMOKE_TEST else "epoch",
    save_strategy="steps" if SMOKE_TEST else "epoch",
    eval_steps=SMOKE_STEPS if SMOKE_TEST else None,
    save_steps=SMOKE_STEPS if SMOKE_TEST else 500,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=1 if SMOKE_TEST else 50,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    report_to="none",
    push_to_hub=False,
    seed=SEED,
    data_seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
)
train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss
1,1.645900,1.296006
2,1.552900,1.257064
3,1.409900,1.248716


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Best checkpoint: ./t5-small-gyafc-formal-checkpoints/checkpoint-19608
Best validation loss: 1.248715877532959


## 7. Evaluate on held-out test data
Compute test loss only after checkpoint selection. This measures how well the model predicts one reference rewrite; it does **not** independently measure formality gain or meaning preservation. Section 9 scores against all four references, and the samples in section 8 are worth reading.

Two caveats on these losses. They score only reference `LABEL_REFERENCE_INDEX`, so never compare runs that differ in that setting. And training targets are a single noisy crowdsourced rewrite each, while validation and test references were curated — so training loss and validation loss are measured against different quality distributions and are less comparable than they were for ParaDetox.

In [8]:
test_metrics = trainer.evaluate(eval_dataset=tokenized["test"], metric_key_prefix="test")
print(test_metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_metrics("test", test_metrics)
if SMOKE_TEST:
    print("Smoke-test results only: these are not full-training quality measurements.")

{'test_loss': 1.156115174293518, 'test_runtime': 7.2363, 'test_samples_per_second': 376.02, 'test_steps_per_second': 47.123, 'epoch': 3.0}


## 8. Save and reload the model
Export the best model and its tokenizer. Reload from the saved directory to verify that inference works without the Trainer. The helper below handles one string at a time and returns an empty string for blank input. Longer inputs are truncated to the configured token limit.

In [9]:
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Reload on CPU to avoid holding a second model copy in GPU memory.
# For faster inference in a separate session, move this model to "cuda".
inference_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR)
inference_model = AutoModelForSeq2SeqLM.from_pretrained(EXPORT_DIR).to("cpu")
inference_model.eval()

@torch.inference_mode()
def formalize(text):
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if not text.strip():
        return ""
    inputs = inference_tokenizer(
        PREFIX + text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True
    ).to(inference_model.device)
    output_ids = inference_model.generate(
        **inputs, max_new_tokens=MAX_LENGTH, num_beams=4, do_sample=False
    )
    return inference_tokenizer.decode(output_ids[0], skip_special_tokens=True)

assert formalize("   ") == ""
print("Saved and reloaded:", Path(EXPORT_DIR).resolve())

Saved and reloaded: /content/t5-small-gyafc-formal


## 9. Inspect rewrites and try your own text
Compare input, references, and model output. Look for a more formal register and preservation of the original meaning. All four human references are shown, and they disagree with each other — that is the point, since there is no single correct formal rewrite. References and model outputs can both be imperfect. The first five test examples are a qualitative check, not a full quality evaluation.

In [10]:
for example in splits["test"].select(range(min(5, len(splits["test"])))):
    print("INPUT: ", example[SOURCE_COLUMN])
    for index, reference in enumerate(example["references"]):
        print(f"REF{index}:  ", reference)
    print("MODEL: ", formalize(example[SOURCE_COLUMN]))
    print()

print("Your example:", formalize("gonna need that report asap, u totally dropped the ball on this one"))

INPUT:  Is Any Baby Really A Freak.
REF0:   Is any baby really a freak.
REF1:   Is any baby really a freak?
REF2:   Is Any Baby really strange?
REF3:   Is anyone really that crazy?
MODEL:  Is any baby really a feak?

INPUT:  aspen colorado has he best music festivals, you sit all over the moutians its  on and just hang out
REF0:   Aspen Colorado has he best music festivals, you sit all over the mountains it is on and just hang out.
REF1:   Aspen, CO has the best music festivals. You can sit on the mountains where they're held and simply enjoy.
REF2:   Aspen Colorado is the home to some great music festivals, you can sit back enjoy the mountains and the music.
REF3:   Aspen, Colorado has great music festivals. You sit over the mountains, and relax.
MODEL:  Aspen Colorado has the best music festivals. You sit all over the moutians. It is on and just hang out.

INPUT:  You can get almost anything on ebay!
REF0:   You can get almost anything on ebay!
REF1:   You can get almost anything on 

## 10. Multi-reference BLEU on the test split
GYAFC's standard metric is corpus BLEU against all four human references, so this cell generates test rewrites with the same decoding settings as `formalize()` and scores them with sacrebleu.

It is kept out of the Trainer deliberately: generating thousands of validation rewrites with beam search after every epoch would dominate the run, and validation loss already selects the checkpoint.

**Read the copy baseline first.** It is the BLEU of the unchanged informal source against the same four references. GYAFC references stay close to their sources, so simply echoing the input scores high. A model BLEU near the copy baseline means the model is barely editing, not that it is good.

BLEU here is pooled across both domains; the GYAFC paper reports them separately. Note also that the batched generator below pads its inputs while `formalize()` does not, so outputs can differ slightly — treat `formalize()` as the reference implementation for the exported model.

In [11]:
import sacrebleu

GENERATION_BATCH_SIZE = 64

@torch.inference_mode()
def generate_all(texts):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = inference_model.to(device)
    outputs = []
    for start in range(0, len(texts), GENERATION_BATCH_SIZE):
        chunk = [PREFIX + text for text in texts[start:start + GENERATION_BATCH_SIZE]]
        inputs = inference_tokenizer(
            chunk, return_tensors="pt", padding=True, max_length=MAX_LENGTH, truncation=True
        ).to(device)
        generated = generator.generate(
            **inputs, max_new_tokens=MAX_LENGTH, num_beams=4, do_sample=False
        )
        outputs += inference_tokenizer.batch_decode(generated, skip_special_tokens=True)
        print(f"  generated {min(start + GENERATION_BATCH_SIZE, len(texts)):,}/{len(texts):,}", flush=True)
    inference_model.to("cpu")
    return [text.strip() for text in outputs]

sources = [text.strip() for text in splits["test"][SOURCE_COLUMN]]
reference_streams = [
    [row[index] for row in splits["test"]["references"]] for index in range(REFERENCE_COUNT)
]
predictions = generate_all(sources)

model_bleu = sacrebleu.corpus_bleu(predictions, reference_streams)
copy_bleu = sacrebleu.corpus_bleu(sources, reference_streams)
unchanged = sum(1 for source, prediction in zip(sources, predictions) if source == prediction)

print(f"\nTest sentences:      {len(sources):,}")
print(f"Model BLEU (4 refs): {model_bleu.score:.2f}")
print(f"Copy BLEU (4 refs):  {copy_bleu.score:.2f}   <- echoing the input scores this")
print(f"BLEU over copy:      {model_bleu.score - copy_bleu.score:+.2f}")
print(f"Outputs identical to input: {unchanged:,} ({unchanged / len(sources):.1%})")
if SMOKE_TEST:
    print("Smoke-test results only: these are not full-training quality measurements.")

  generated 64/2,721
  generated 128/2,721
  generated 192/2,721
  generated 256/2,721
  generated 320/2,721
  generated 384/2,721
  generated 448/2,721
  generated 512/2,721
  generated 576/2,721
  generated 640/2,721
  generated 704/2,721
  generated 768/2,721
  generated 832/2,721
  generated 896/2,721
  generated 960/2,721
  generated 1,024/2,721
  generated 1,088/2,721
  generated 1,152/2,721
  generated 1,216/2,721
  generated 1,280/2,721
  generated 1,344/2,721
  generated 1,408/2,721
  generated 1,472/2,721
  generated 1,536/2,721
  generated 1,600/2,721
  generated 1,664/2,721
  generated 1,728/2,721
  generated 1,792/2,721
  generated 1,856/2,721
  generated 1,920/2,721
  generated 1,984/2,721
  generated 2,048/2,721
  generated 2,112/2,721
  generated 2,176/2,721
  generated 2,240/2,721
  generated 2,304/2,721
  generated 2,368/2,721
  generated 2,432/2,721
  generated 2,496/2,721
  generated 2,560/2,721
  generated 2,624/2,721
  generated 2,688/2,721
  generated 2,721/2,721

## 11. Download the model
Archive the exported folder and download it. Keep `PREFIX = "formalize: "` and `MAX_LENGTH = 64` with the model: inference must use the same prefix string, and the model has no training signal for inputs longer than about 58 tokens.

In [12]:
export_path = Path(EXPORT_DIR).resolve()
archive_path = shutil.make_archive(
    str(export_path), "zip", root_dir=export_path.parent, base_dir=export_path.name
)
print("Model archive:", archive_path)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: copy the ZIP from the path above.")
else:
    files.download(archive_path)

Model archive: /content/t5-small-gyafc-formal.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>